# Check a core comparison before interpreting the score

Run this with the project's Python 3.10 Jupyter kernel. The invented CSV
contains one training tube and one held tube, with two predictions for the
same held interval. The CSV already contains projected RLD. The notebook
illustrates quality checks and score interpretation with a tiny fixture.

In [ ]:
import os
from pathlib import Path

import pandas as pd

from deeprootgen.form.core_comparison import compare_core_csv

here = Path.cwd()
configured_root = os.environ.get("DRG_PROJECT_ROOT")
root = Path(configured_root) if configured_root else next(
    folder for folder in (here, *here.parents)
    if (folder / "examples/research/core_comparison_synthetic.csv").is_file()
)
source = root / "examples/research/core_comparison_synthetic.csv"
observed = pd.read_csv(source)
training = observed.loc[observed["split"] == "train"]
held = observed.loc[observed["split"] == "held"]
section = ["core_id", "z_lower_cm", "z_upper_cm"]
assert set(training["core_id"]).isdisjoint(held["core_id"])
assert (observed["z_upper_cm"] > observed["z_lower_cm"]).all()
assert (observed["core_volume_cm3"] > 0).all()
assert held["predicted_rld"].notna().all()
print(f"Training tubes = {training['core_id'].nunique()}")
print(f"Held tubes = {held['core_id'].nunique()}")
print(f"Distinct held sections = {len(held.drop_duplicates(section))}")
observed

## Score the candidates

The training tube sets the normalization scale at **0.20 cm/cm³** of root length density (RLD). The held section has **0.10 cm/cm³**. The `exact` candidate predicts 0.10 and scores **0.00**. The `miss` candidate predicts 0.00. Its loss is the sum of three weighted terms.

| Component | Calculation for `miss` | Contribution |
|---|---|---:|
| Normalized absolute error (`nmae`) | `abs(0.10 - 0.00) / 0.20 = 0.50` | `0.50 × 0.50 = 0.25` |
| Tube-by-depth total variation (`joint_tv`) | `1.00` across the held tube sections | `0.25 × 1.00 = 0.25` |
| Depth total variation (`depth_tv`) | `1.00` after pooling by depth | `0.25 × 1.00 = 0.25` |
| **Composite loss** | | **0.75** |

Both total variation terms are 1.00 because the observed profile contains roots and the predicted profile is entirely empty. In this one-section example, pooling by depth gives the same result as comparing tube sections.

The `miss` and `false_hit` columns are separate detection diagnostics. At the default threshold of zero, `miss = 1.0` for the `miss` candidate means it missed the only detected held section. `false_hit` is `None` for both candidates because the held observation contains no section classified as empty.

In [ ]:
rows, train_tubes, held_tubes = compare_core_csv(source.read_text())
assert (train_tubes, held_tubes) == (1, 1)
assert (rows[0]["candidate"], rows[0]["loss"]) == ("exact", 0.0)
assert (rows[1]["candidate"], rows[1]["loss"], rows[1]["miss"]) == ("miss", 0.75, 1.0)
pd.DataFrame(rows)

## Check boundaries

### 1. Reject a tube that appears in both splits

The code copies the input table and replaces the training tube's `core_id` with the held tube's ID. The `split` labels still say `train` and `held`, but the scorer now sees the same physical tube in both groups. It checks the IDs before scoring and raises `ValueError`. The test checks the error message and fails if the bad split is accepted. Keeping every interval from a tube in the same split makes the score a test on a tube kept out of training.

### 2. Raise the detection threshold

An observed or predicted RLD counts as a hit when it is **strictly greater than** the threshold. At zero, the observed 0.10 is a hit. The `exact` candidate also predicts a hit (0.10), while `miss` predicts an empty section (0.00).

At **0.15 cm/cm³**, the observed 0.10 and both predictions fall below the threshold. The miss rate is `None` for both candidates because there are zero observed hits in its denominator. The false-hit rate is `0.0` for both because there is one observed section classified as empty and neither candidate predicts a hit there.

The continuous loss still uses the original RLD values, so the scores stay **0.00** and **0.75**. The test checks both the unchanged losses and the undefined miss rates. Choose a threshold supported by the measurement method in a real study.

In [ ]:
leaked = observed.copy()
leaked.loc[leaked["split"] == "train", "core_id"] = held["core_id"].iloc[0]
try:
    compare_core_csv(leaked.to_csv(index=False))
except ValueError as error:
    assert "Hold out entire tubes" in str(error)
    print(f"Overlapping tube rejected. {error}")
else:
    raise AssertionError("Expected an overlapping-tube error.")

threshold_rows, _, _ = compare_core_csv(source.read_text(), threshold_rld=0.15)
assert [row["loss"] for row in threshold_rows] == [row["loss"] for row in rows]
assert all(row["miss"] is None for row in threshold_rows)
pd.DataFrame(threshold_rows)[["candidate", "loss", "miss", "false_hit"]]

## Apply the lesson

Upload the same invented CSV to Dash **Core Comparison** and compare the
displayed values. For field work, check the sampling geometry and align
projected RLD before using the scorer. Reserve whole tubes before selecting
a model or fitting a spatial surface. This fixture demonstrates scoring
behavior. Assess transfer on field data with independent positions.

### From this exercise to 14 field cores

| Stage | Which cores supply the prediction errors? |
|---|---|
| Fit and select settings | Use 11 development cores. To check transfer during development, hold out whole tubes in turn within those 11. |
| Evaluate the selected settings | Use the 3 reserved cores. Their measured and predicted RLD supply the errors and both distribution terms. The mean measured RLD from the 11 development cores still sets the normalization scale. |
| Report site-wide agreement (optional) | Use all 14 cores with the same fixed settings and development scale. This descriptive score includes the positions used for fitting. Recompute the distribution terms across all 14 because they depend on the pooled RLD shares. |

Choose the reserved tubes before inspecting model errors. Cover the along-row positions, distances from the row, and sampled depths that matter for the prediction. Apply the same quality checks in both sets, retain valid measured zeros, and exclude unmeasured intervals from scoring. If the goal is prediction for new vines, reserve whole vines.

[Fresh-data guide](FRESH_DATA_WITH_CLAUDE.md) ·
[Scoring definitions](../../docs/ARCHITECTURE_SCORING.md)
### Diagnose these cases before scoring a new field table

- Positive-down depths copied into the elevation columns need a coordinate
  conversion from the declared soil surface.
- A missing interval filled with zero would be counted as a genuine
  field-zero section. Recover its sampling status from the source.
- Shuffled prediction rows need a verified join by interval key.
- A prediction for a different diameter band measures different roots.
- Different tubes around the same vine need to stay together when the
  evaluation asks about transfer to new vines.
